# 서비스오더 데이터 분석 (실무 수동검증 특화 버전)

1. **캐시 로드 및 연월 복원**: `서비스오더.pkl` 또는 `23_26년_서비스오더.pkl`을 1초 만에 불러와 `시트명` 기반으로 연월 정보를 100% 무결하게 복구합니다.
2. **H051 -> H073 전처리**: 센터 코드를 하나로 병합합니다.
3. **전체오더 집계 (5단계)**: 지정된 소분류 6가지 기준의 월별/사업부별 전체오더 집계표 작성 및 엑셀 출력 (`전체오더_월별집계.xlsx`).
4. **신규 아파트 분류 및 인터넷 복사 검증 패키지 (6단계 세분화 셀)**:
   * **자동 필터**: 소분류가 `전입신청`이고, 동일 년도월 동일 주소에서 **15건 이상** 발생하는 신규 단지 후보군 자동 선별 (키워드 룰 제거).
   * **복사용 가이드 제공**: 선별된 대형 지점 주소들을 **마우스 드래그 복사(Ctrl+C)가 극도로 편리한 텍스트 폼**으로 화면에 쫙 뿌려주어, 네이버 부동산에 즉시 조회할 수 있도록 실무 편의를 지원합니다.

## 1단계: 라이브러리 임포트 및 파일 경로 정의

In [1]:
import pandas as pd
import shutil
import re
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

In [2]:
# 1. 원본 DocuONE 네트워크 pkl 경로
ORIGINAL_PKL_PATH = Path(r"\\DocuONE\\MyDrive\\개인함\\23_26년_서비스오더\\서비스오더.pkl")

# 2. 로컬 캐시 파일 경로
LOCAL_CACHE_1 = Path("23_26년_서비스오더.pkl")
LOCAL_CACHE_2 = Path("서비스오더.pkl")

print(f"네트워크 원본 pkl: {ORIGINAL_PKL_PATH}")

네트워크 원본 pkl: \\DocuONE\\MyDrive\개인함\23_26년_서비스오더\서비스오더.pkl


## 2단계: 데이터 다이렉트 로드 (고속 캐싱 로드)

In [3]:
# 1. 로컬에 존재하는 적절한 pkl 파일 탐색
path_to_read = None

if LOCAL_CACHE_1.exists():
    path_to_read = LOCAL_CACHE_1
elif LOCAL_CACHE_2.exists():
    path_to_read = LOCAL_CACHE_2
else:
    # 로컬에 아무것도 없다면 네트워크에서 로컬로 고속 복사 후 캐싱
    if ORIGINAL_PKL_PATH.exists():
        print(f"최초 1회 로컬 캐시 복사를 시작합니다... (약 600MB)")
        shutil.copy(ORIGINAL_PKL_PATH, LOCAL_CACHE_2)
        print("로컬 복사 완료!")
        path_to_read = LOCAL_CACHE_2
    else:
        print(f"경고: 원본 네트워크 파일({ORIGINAL_PKL_PATH})에 접근할 수 없습니다.")

# 2. 데이터 초고속 로드 실행
if path_to_read and path_to_read.exists():
    print(f"데이터를 pickle 방식으로 초고속 로드하는 중... ({path_to_read.name})")
    df = pd.read_pickle(path_to_read)
    print("데이터 로드 성공!")
    print(f"- 전체 행(Row) 수: {len(df):,}개")
    print(f"- 컬럼 구성: {list(df.columns)}")
else:
    print("데이터 로드에 실패했습니다. 파일 경로를 다시 확인해 주세요.")
    df = None

데이터를 pickle 방식으로 초고속 로드하는 중... (서비스오더.pkl)
데이터 로드 성공!
- 전체 행(Row) 수: 1,833,764개
- 컬럼 구성: ['서비스처리센터', '고객방문일', '고객방문시간', '우선순위', '오더번호', '상태', '대분류', '중분류', '소분류', '내역', '세대', '고객서비스처리센터', '고객번호', '계약계정번호', '정보', '정보(BP)', '최초공급일', '주소', '건물동', '호수', '전입/전출일', '오더생성일', '오더생성시간', '오더생성자', '부서확인일', '부서확인시간', '부서확인자', '기사배정일', '기사배정시간', '배정된기사', '마감일', '마감시간', '마감자', '변경일', '변경시간', '변경자', '수수료금액', '영수증번호', '행정동', '시트명', 'Unnamed: 10', '세대.1', '설치유형', '설치유형 Text']


## 3단계: 시트명 기반 년도월 컬럼 자동 복원

In [4]:
def extract_ym_from_sheet(sheet_name: str) -> str:
    """시트명 속에 들어있는 (YY.MM 형태의 연월 정보를 정확하게 추출하여 YYYY-MM 포맷으로 반환합니다."""
    m = re.search(r"\((\d{2})\.(\d{1,2})", str(sheet_name).strip())
    if m:
        yy, mm = int(m.group(1)), int(m.group(2))
        return f"20{yy:02d}-{mm:02d}"
    return ""

if df is not None:
    # 컬럼 공백 정리
    df.columns = [str(col).strip() for col in df.columns]
    
    if "시트명" in df.columns:
        print("시트명을 기반으로 년도월 컬럼 생성을 실행합니다.")
        df["년도월"] = df["시트명"].apply(extract_ym_from_sheet)
        print(f"생성 성공! 연월 고유값 샘플: {sorted(df['년도월'].dropna().unique())}")
    else:
        print("경고: 데이터에 '시트명' 컬럼이 존재하지 않아 년도월을 복원할 수 없습니다.")

시트명을 기반으로 년도월 컬럼 생성을 실행합니다.
생성 성공! 연월 고유값 샘플: ['2023-01', '2023-02', '2023-03', '2023-04', '2023-05', '2023-06', '2023-07', '2023-08', '2023-09', '2023-10', '2023-11', '2023-12', '2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06', '2024-07', '2024-08', '2024-09', '2024-10', '2024-11', '2024-12', '2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06', '2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12', '2026-01', '2026-02', '2026-03', '2026-04']


## 4단계: 데이터 전처리 (서비스처리센터 H051 -> H073 병합)

In [5]:
if df is not None:
    # 서비스처리센터 공백 정리
    df["서비스처리센터"] = df["서비스처리센터"].fillna("").astype(str).str.strip()
    
    print("=== 변경 전 서비스처리센터 분포 ===")
    print(df["서비스처리센터"].value_counts(dropna=False).head(5))
    
    # H051 -> H073 치환 실행
    df["서비스처리센터"] = df["서비스처리센터"].replace({"H051": "H073"})
    
    print("\n=== 변경 후 서비스처리센터 분포 ===")
    print(df["서비스처리센터"].value_counts(dropna=False).head(5))
    
    # 검증
    h051_count = (df["서비스처리센터"] == "H051").sum()
    print(f"\n치환 완료 후 H051 남은 데이터 건수: {h051_count}개")

=== 변경 전 서비스처리센터 분포 ===
서비스처리센터
H074    409326
H075    359545
H073    359189
H072    339794
H071    319534
Name: count, dtype: int64

=== 변경 후 서비스처리센터 분포 ===
서비스처리센터
H074    409326
H073    405535
H075    359545
H072    339794
H071    319534
Name: count, dtype: int64

치환 완료 후 H051 남은 데이터 건수: 0개


## 5단계: 전체오더 월별/사업부별 집계표 생성 및 엑셀 출력

* **대상 소분류 6가지**: `전입신청`, `전입변경`, `전출신청`, `전출변경`, `연소기 연결/교체`, `연소기철거`

In [6]:
ALL_TARGET_CATEGORIES = [
    "전입신청",
    "전입변경",
    "전출신청",
    "전출변경",
    "연소기 연결/교체",
    "연소기철거"
]

if df is not None and "년도월" in df.columns:
    # 1. 소분류 6개 필터링
    df_total_orders = df[df["소분류"].isin(ALL_TARGET_CATEGORIES)].copy()
    print(f"소분류 6개 전체오더 필터링 완료 건수: {len(df_total_orders):,}개")
    
    # 2. 텍스트 정제
    for col in ["서비스처리센터", "행정동", "년도월"]:
        df_total_orders[col] = df_total_orders[col].fillna("").astype(str).str.strip()
        
    # 빈값 및 nan 제외
    df_total_orders_clean = df_total_orders[
        (df_total_orders["서비스처리센터"] != "") &
        (df_total_orders["행정동"] != "") &
        (df_total_orders["년도월"] != "") &
        (df_total_orders["서비스처리센터"] != "nan") &
        (df_total_orders["행정동"] != "nan")
    ]
    
    # 3. 월별 오더 건수 집계표 생성
    total_order_matrix = (
        df_total_orders_clean.groupby(["서비스처리센터", "행정동", "년도월"])
        .size()
        .unstack(fill_value=0)
    )
    
    # 4. 행별/열별 합계 얹기
    total_order_matrix["합계"] = total_order_matrix.sum(axis=1)
    
    total_row = total_order_matrix.sum(axis=0).to_frame().T
    total_row.index = pd.MultiIndex.from_tuples([("전체", "총합")], names=["서비스처리센터", "행정동"])
    
    final_total_matrix = pd.concat([total_order_matrix, total_row]).astype(int)
    
    print("\n=== 전체오더 월별/사업부별 집계표 (상위 20개 행) ===")
    display(final_total_matrix.head(20))
    
    # 5. 엑셀 파일 출력
    # total_excel_path = Path("전체오더_월별집계.xlsx")
    # final_total_matrix.to_excel(total_excel_path)
    # print(f"\n전체오더 집계표가 엑셀 파일로 저장되었습니다: {total_excel_path.absolute()}")

소분류 6개 전체오더 필터링 완료 건수: 1,123,695개

=== 전체오더 월별/사업부별 집계표 (상위 20개 행) ===


년도월          2023-01  2023-02  2023-03  2023-04  2023-05  2023-06  2023-07  \
서비스처리센터 행정동                                                                  
H071    가정동       20       12        8        6        9        9       11   
        갈마동     1764     2103     1684     1520     1335     1334     1430   
        도룡동      261      312      171      164      149      169      168   
        둔산동     1172     1445      993      780      776      771      873   
        만년동      402      499      309      259      258      247      341   
        어은동        0        1        0        0        0        0        0   
        용문동        0        1        0        0        0        0        0   
        월평동      949     1192      927      774      680      735      761   
        탄방동      882     1115      957      827      743      673      784   
H072    가정동        0        0        0        0        0        0        0   
        관평동      330      332      237      270      337      327      273   
        구룡동       31       63       33       45       49       44       39   
        대화동       90       69       93      116       64       72       84   
        덕암동      157      207      227      188      158      190      156   
        덕진동        0        0        0        0        0        0        0   
        도룡동        0        0        0        0        0        0        0   
        둔곡동      528      266      208       96       73       45       65   
        목상동      113      145      150      139      148      118      109   
        문지동       92       80      116       82       87       88      112   
        문평동        0        0        2        0        0        0        0   

년도월          2023-08  2023-09  2023-10  2023-11  2023-12  2024-01  2024-02  \
서비스처리센터 행정동                                                                  
H071    가정동        4        4        4        2        7        9        9   
        갈마동     1424     1210     1473     1549     1778     1736     1812   
        도룡동      155       98      194      174      245      222      241   
        둔산동      816      714      813      931     1382     1332     1380   
        만년동      328      180      223      219      378      393      513   
        어은동        0        0        0        0        0        0        0   
        용문동        0        0        0        0        0        0        0   
        월평동      763      626      927      769      911     1267     1029   
        탄방동      629      735      744      784      951     1045      991   
H072    가정동        0        0        0        0        0        0        0   
        관평동      275      219      253      218      272      314      288   
        구룡동       13       20       27       12       20       32       14   
        대화동       71       52       88       92       69       69       63   
        덕암동      127      136      155      223      213      171      199   
        덕진동        0        0        0        0        0        0        0   
        도룡동        0        0        0        0        0        0        0   
        둔곡동       24       57       46      132       60       52       65   
        목상동      133      100      124      174      153      179      171   
        문지동       62       55       64       67       85       89       83   
        문평동        0        0        0        0        0        2        0   

년도월          2024-03  2024-04  2024-05  2024-06  2024-07  2024-08  2024-09  \
서비스처리센터 행정동                                                                  
H071    가정동        5        3        9        1       10        8        7   
        갈마동     1710     1455     1427     1371     1347     1309     1240   
        도룡동      239      154      149      154      179      150      107   
        둔산동     1203      740      754      798      805      873      742   
        만년동      278      242      228      235      319      313      210   
        어은동        0

## 6단계: 신규 아파트 수동검증 특화 필터링 엔진

### 6-1단계: '전입신청' 데이터 추출 및 주소 텍스트 정제

In [ ]:
if df is not None:
    # 1. 소분류 = '전입신청' 데이터만 필터링
    df_move_in = df[df["소분류"] == "전입신청"].copy()
    print(f"전입신청 전체 데이터 수: {len(df_move_in):,}개")
    
    # 2. 텍스트 공백 정제
    for col in ["서비스처리센터", "행정동", "주소", "년도월"]:
        df_move_in[col] = df_move_in[col].fillna("").astype(str).str.strip()
        
    # 3. 주소 쪼개짐 방지 (단지 꼬리표 쳐내기 정규화)
    # 예: "둔산더샵엘리프 1단지", "둔산더샵엘리프 2단지" -> "둔산더샵엘리프"로 그룹바이가 묶이도록 정규화
    def clean_address_tail(addr_str):
        addr_clean = re.sub(r"\s+\d+단지\b", "", addr_str)
        addr_clean = re.sub(r"\b(\d+)단지(?=\s*[\)])", "", addr_clean)
        addr_clean = re.sub(r"\b(써밋그랜드파크|플레이스|더샵|엘리프|휴리움|써밋|그랜드파크)\s+\d+(?=\s*[\)])", r"\1", addr_clean)
        return addr_clean.strip()
        
    df_move_in["주소_원본"] = df_move_in["주소"]
    df_move_in["주소"] = df_move_in["주소"].apply(clean_address_tail)
    print("주소 쪼개짐 방지 (단지 정규화) 적용 완료!")
    
    # 4. 외부 '확정_아파트_리스트.txt' 파일에서 확실한 아파트 고유 주소 리스트 동적 로드
    master_list_path = Path("apartments.txt")
    MASTER_APARTMENTS = set()
    if master_list_path.exists():
        with open(master_list_path, "r", encoding="utf-8") as f:
            MASTER_APARTMENTS = set([line.strip() for line in f if line.strip()])
        print(f"성공: 외부 확정 아파트 목록 로딩 완료 ({len(MASTER_APARTMENTS)}개 단지)")
    else:
        print("경고: apartments.txt 파일이 존재하지 않습니다.")

전입신청 전체 데이터 수: 472,260개
주소 쪼개짐 방지 (단지 정규화) 적용 완료!
성공: 외부 확정 아파트 목록 로딩 완료 (761개 단지)


### 6-2단계: [규모 검증] 동일 월 동일 주소에서 15건 이상 발생하는 후보 단지만 추출

동일 년도월 동일 주소에서 전입신청이 **15건 이상** 발생하는 대형 단지 후보군만 선별하고, 15건 미만인 소형 주소지는 예외 없이 모두 드롭합니다.

In [8]:
if 'df_move_in' in locals():
    # 1. 동일 년도월 및 동일 주소의 전입신청 빈도(건수) 계산
    df_move_in["addr_count"] = df_move_in.groupby(["년도월", "주소"])["주소"].transform("size")
    
    # 2. 15건 이상 데이터만 필터링
    df_large_candidates = df_move_in[df_move_in["addr_count"] >= 15].copy()
    
    # 3. 마스터 아파트 주소 리스트에 부합하는 데이터만 선별 (정제 전/후 주소 크로스 체킹)
    df_new_apt_final = df_large_candidates[
        (df_large_candidates["주소_원본"].isin(MASTER_APARTMENTS)) |
        (df_large_candidates["주소"].isin(MASTER_APARTMENTS))
    ].copy()
    
    print(f"[정밀 필터 완료] 동일 월 15건 이상 대형 신규 아파트 실 데이터 수: {len(df_new_apt_final):,}개")
    print(f"- 필터 통과 고유 신규 아파트 단지 수: {df_new_apt_final['주소'].nunique():,}개")

[정밀 필터 완료] 동일 월 15건 이상 대형 신규 아파트 실 데이터 수: 42,022개
- 필터 통과 고유 신규 아파트 단지 수: 206개


### 6-3단계: 🌟 인터넷 검색 검증용 주소 복사용 가이드 출력 (마우스 복사 특화)

15건 이상 전입신청이 폭증하여 검출된 고유 주소(단지 후보군) 리스트를 마우스 드래그 복사가 매우 편리한 텍스트 폼으로 출력합니다. 

**👉 사용법**: 아래의 `주소 복사용:` 텍스트를 마우스로 긁어 **네이버 부동산**이나 **인터넷등기소**에 검색하여 오피스텔(비주택)로 판정되면 `오피스텔_제외_리스트`에 등록해 관리합니다.

In [9]:
if 'df_new_apt_final' in locals():
    # 최종 신규 아파트로 분류되어 무결하게 매칭된 결과 요약 출력
    candidates = (
        df_new_apt_final.groupby(["년도월", "서비스처리센터", "행정동", "주소"])
        .size()
        .reset_index(name="전입신청건수")
    )
    candidates = candidates.sort_values(by=["년도월", "전입신청건수"], ascending=[True, False]).reset_index(drop=True)
    
    print(f"=== 최종 검증 및 정제 완료된 신규 아파트 오더 매칭 목록 (총 {len(candidates)}개 단지) ===\n")
    
    current_ym = ""
    for idx, row in candidates.iterrows():
        if row["년도월"] != current_ym:
            current_ym = row["년도월"]
            print(f"\n━━━━━━━━━━━━━━━━━━━━━━━━ {current_ym} 월 신규 입주 아파트 단지 ━━━━━━━━━━━━━━━━━━━━━━━━")
            
        print(f"[{idx+1}] 확정건수: {row['전입신청건수']}건 | 센터: {row['서비스처리센터']} | 행정동: {row['행정동']}")
        print(f"    주소명: {row['주소']}\n")

=== 최종 검증 및 정제 완료된 신규 아파트 오더 매칭 목록 (총 1101개 단지) ===


━━━━━━━━━━━━━━━━━━━━━━━━ 2023-01 월 신규 입주 아파트 단지 ━━━━━━━━━━━━━━━━━━━━━━━━
[1] 확정건수: 176건 | 센터: H072 | 행정동: 둔곡동
    주소명: 과학성장로 77 (둔곡동 430 서한이다음1단지)

[2] 확정건수: 175건 | 센터: H072 | 행정동: 둔곡동
    주소명: 과학성장로 33 (둔곡동 432 서한이다음2단지)

[3] 확정건수: 62건 | 센터: H073 | 행정동: 두마면
    주소명: 두마면 농소로 63 (농소리 973 계룡푸르지오더퍼스트)

[4] 확정건수: 51건 | 센터: H074 | 행정동: 용운동
    주소명: 용운로 203 (용운동 783 용운e편한세상대전에코포레아파트)

[5] 확정건수: 42건 | 센터: H072 | 행정동: 둔곡동
    주소명: 과학성장로 80 (둔곡동 431 우미린)

[6] 확정건수: 37건 | 센터: H073 | 행정동: 엄사면
    주소명: 엄사면 번영3길 73-12 (엄사리 226-4 삼진아파트)

[7] 확정건수: 32건 | 센터: H073 | 행정동: 신도안면
    주소명: 신도안면 신도안3길 55 (남선리 1340 계룡대해미르아파트)

[8] 확정건수: 31건 | 센터: H073 | 행정동: 엄사면
    주소명: 엄사면 엄사중앙로 66 (엄사리 282 성원아파트)

[9] 확정건수: 31건 | 센터: H075 | 행정동: 추목동
    주소명: 자운로97번길 431 (추목동 584 육대아파트)

[10] 확정건수: 27건 | 센터: H072 | 행정동: 송강동
    주소명: 송강로42번길 61 (송강동 8-2 송강청솔아파트)

[11] 확정건수: 26건 | 센터: H072 | 행정동: 용전동
    주소명: 용전동 143-59

[12] 확정건수: 25건 | 센터: H075 | 행정동: 지족동
    주소명: 노은서로2

### 6-4단계: 신규 아파트 월별/사업부별 최종 집계표 작성 및 엑셀 출력

In [10]:
if 'df_new_apt_final' in locals():
    # 빈값 제외 및 정제
    df_matrix_data = df_new_apt_final[
        (df_new_apt_final["서비스처리센터"] != "") &
        (df_new_apt_final["행정동"] != "") &
        (df_new_apt_final["년도월"] != "") &
        (df_new_apt_final["서비스처리센터"] != "nan") &
        (df_new_apt_final["행정동"] != "nan")
    ].copy()
    
    # 1. 집계표 산출 (Groupby & Size & Unstack)
    apt_matrix = (
        df_matrix_data.groupby(["서비스처리센터", "행정동", "년도월"])
        .size()
        .unstack(fill_value=0)
    )
    
    # 2. 행별 합계 및 열별 총계 산출
    
    apt_matrix["합계"] = apt_matrix.sum(axis=1)
    
    total_row = apt_matrix.sum(axis=0).to_frame().T
    total_row.index = pd.MultiIndex.from_tuples([("전체", "총합")], names=["서비스처리센터", "행정동"])
    
    # 3. 결합 및 정수형 변환
    final_apt_matrix = pd.concat([apt_matrix, total_row]).astype(int)
    
    print("=== 신규아파트 월별/사업부별 1차 후보군 집계표 (상위 20개 행) ===")
    display(final_apt_matrix.head(20))
    
    # 4. 실무 보고용 엑셀 출력
    # output_excel = Path("신규아파트_월별집계(주소).xlsx")
    # final_apt_matrix.to_excel(output_excel)
    # print(f"\n신규아파트 1차 후보군 집계표가 엑셀 파일로 저장되었습니다: {output_excel.absolute()}")

=== 신규아파트 월별/사업부별 1차 후보군 집계표 (상위 20개 행) ===


년도월           2023-01  2023-02  2023-03  2023-04  2023-05  2023-06  2023-07  \
서비스처리센터 행정동                                                                   
H071    갈마동         0       22       17       15        0       16       16   
        도룡동         0       32        0        0        0        0        0   
        둔산동        59       52       35       15        0       17       66   
        만년동        32        0        0        0       19        0       16   
        월평동         0        0        0        0       18        0        0   
        탄방동        16       15        0        0        0       15        0   
H072    덕암동         0        0       23        0        0        0        0   
        둔곡동       393      160      131       21       19        0        0   
        목상동         0        0        0        0       16        0        0   
        문지동         0       21        0       15       25       15       24   
        법동         18       43        0        0        0       33        0   
        봉산동         0        0        0        0        0        0        0   
        석봉동         0        0        0       17       19       25       15   
        송강동        27       22       26       61       29       23       44   
        송촌동         0        0       20        0        0        0        0   
        신대동         0        0        0        0        0        0        0   
        신탄진동        0        0        0        0        0        0        0   
        연축동         0        0        0        0        0        0        0   
        와동          0        0        0        0        0        0        0   
        용산동         0        0        0        0        0        0        0   

년도월           2023-08  2023-09  2023-10  2023-11  2023-12  2024-01  2024-02  \
서비스처리센터 행정동                                                                   
H071    갈마동         0        0       19        0        0        0       30   
        도룡동         0        0       36        0        0        0        0   
        둔산동        42       26       54        0      147      133      122   
        만년동        17        0       16       17       22        0       17   
        월평동         0        0      115        0       16      110        0   
        탄방동         0        0        0        0       19       21       16   
H072    덕암동         0        0        0       33        0        0        0   
        둔곡동         0        0        0        0        0        0        0   
        목상동         0        0        0        0        0        0        0   
        문지동         0        0       16        0        0       20       18   
        법동         26        0        0        0      101       35        0   
        봉산동         0        0        0        0       19        0        0   
        석봉동         0        0        0       24       18        0        0   
        송강동        29       27       27       19      107       60       22   
        송촌동         0        0        0        0        0        0        0   
        신대동         0        0        0        0        0       15        0   
        신탄진동        0        0        0        0       28       27      132   
        연축동         0        0        0        0        0        0        0   
        와동          0        0        0        0        0        0       47   
        용산동         0        0        0        0        0        0        0   

년도월           2024-03  2024-04  2024-05  2024-06  2024-07  2024-08  2024-09  \
서비스처리센터 행정동                                                                   
H071    갈마동        18       20       16       19        0       15        0   
        도룡동         0        0        0        0        0        0        0   
        둔산동        90       22       21       32        0       53        0   
        만년동        15       15       20        0        0        0       23   
        월평동        15        0        0       80

### 7단계: 23년 대비 현행년도까지 신규 아파트 데이터 취합

In [ ]:
# 1. 확정 아파트 목록 로드 (절대경로 confirmed_apartments.txt 참조)
master_list_path = Path(r"c:\DA\apartments.txt")
with open(master_list_path, "r", encoding="utf-8") as f:
    master_addrs = [line.strip() for line in f if line.strip()]

# 중복 도로명 해결을 위한 canonical 주소 매핑 구축 (가장 상세한 주소(긴 문자열)를 대표로 채택)
road_to_canonical = {}
for line in master_addrs:
    road_key = line.split('(')[0].strip()
    if road_key not in road_to_canonical:
        road_to_canonical[road_key] = line
    else:
        if len(line) > len(road_to_canonical[road_key]):
            road_to_canonical[road_key] = line

# 2. 양방향(Dual-Way) 도로명/지번 매핑 테이블 구축 (대표 주소로 매핑 단일화)
road_to_master = {}
for addr in master_addrs:
    road_key = addr.split('(')[0].strip()
    canonical_addr = road_to_canonical[road_key]
    road_to_master[road_key] = canonical_addr
    
    # 지번/동명 키 등록 (괄호 안에서 동명+번지 추출)
    if '(' in addr:
        inner = addr.split('(')[1].split(')')[0].strip()
        m = re.search(r"^([가-힣]+동\s+\d+(?:-\d+)?)", inner)
        if m:
            jibun_key = m.group(1).strip()
            road_to_master[jibun_key] = canonical_addr

# 3. 전체 데이터 중 소분류 = 전입신청, 전입변경인 데이터 필터링
df_filtered = df[df["소분류"].isin(["전입신청", "전입변경"])].copy()
for col in ["서비스처리센터", "행정동", "주소", "년도월"]:
    df_filtered[col] = df_filtered[col].fillna("").astype(str).str.strip()

# 4. 주소 쪼개짐 방지 정규화 및 대표 주소 매핑 생성
def clean_address_tail(addr_str):
    addr_clean = re.sub(r"\s+\d+단지\b", "", addr_str)
    addr_clean = re.sub(r"\b(\d+)단지(?=\s*[\)])", "", addr_clean)
    addr_clean = re.sub(r"\b(써밋그랜드파크|플레이스|더샵|엘리프|휴리움|써밋|그랜드파크)\s+\d+(?=\s*[\)])", r"\1", addr_clean)
    return addr_clean.strip()

df_filtered["주소_정제"] = df_filtered["주소"].apply(clean_address_tail)

# 5. 원본 주소의 괄호 앞부분 추출
df_filtered["주소_키워드"] = df_filtered["주소_정제"].apply(lambda x: str(x).split('(')[0].strip())

# 6. 해시 맵핑을 활용한 1초 초고속 정밀 매칭 (오탐지 0%, 양방향 완벽 대응)
df_filtered["road_part"] = df_filtered["주소_키워드"].map(road_to_master)

df_matched = df_filtered[df_filtered["road_part"].notna()].copy()
print(f"마스터 아파트 목록에 매칭된 전체 전입오더 수: {len(df_matched):,}개")


마스터 아파트 목록에 매칭된 전체 전입오더 수: 147,794개


In [12]:
# 2023년 전체(1~12월) 동안 서비스오더(전입신청, 전입변경)에 한 번이라도 등장했던 아파트 주소 수집 (Baseline)
df_2023 = df_matched[df_matched["년도월"].str.startswith("2023-")].copy()
addresses_2023 = set(df_2023["road_part"].dropna().unique())
print(f"2023년 전체 이력에 등장한 기존 고유 아파트 단지 수: {len(addresses_2023):,}개")


2023년 전체 이력에 등장한 기존 고유 아파트 단지 수: 574개


In [13]:
# 1. 2024년 ~ 2026년 데이터 추출
df_target = df_matched[
    df_matched["년도월"].str.startswith("2024-") |
    df_matched["년도월"].str.startswith("2025-") |
    df_matched["년도월"].str.startswith("2026-")
].copy()

# 2. 월별 동일 주소 전입 건수 15건 이상 집계
monthly_counts = df_target.groupby(["년도월", "road_part"]).size().reset_index(name="count")
large_monthly = monthly_counts[monthly_counts["count"] >= 15].copy()

# 3. 2023년 이력에 없던 진짜 '신규 생성 주소'만 필터링
large_monthly = large_monthly[~large_monthly["road_part"].isin(addresses_2023)].copy()
large_monthly = large_monthly.sort_values(by=["road_part", "년도월"])
large_monthly["구분"] = ""
print(f'2023년 Baseline 필터 통과 후 24~26년 신규 아파트 입주 오더 수: {len(large_monthly)}건')
large_monthly


2023년 Baseline 필터 통과 후 24~26년 신규 아파트 입주 오더 수: 229건


,년도월,road_part,count,구분
8048,2025-06,계룡로 520 (탄방동 둔산자이아이파크),90,
8526,2025-07,계룡로 520 (탄방동 둔산자이아이파크),831,
9014,2025-08,계룡로 520 (탄방동 둔산자이아이파크),700,
9503,2025-09,계룡로 520 (탄방동 둔산자이아이파크),195,
9993,2025-10,계룡로 520 (탄방동 둔산자이아이파크),48,
...,...,...,...,...
9474,2025-08,탄방로 132 (용문동 둔산더샵엘리프),19,
10935,2025-11,탄방로 132 (용문동 둔산더샵엘리프),15,
456,2024-01,태평동 334-15,42,
3222,2024-07,홍도동 65-2,46,


In [14]:
# 동일 주소 내에서 가장 빠른 발생 월을 '최초', 이후 발생 월을 '지속'으로 규정
for road in large_monthly["road_part"].unique():
    road_indices = large_monthly[large_monthly["road_part"] == road].index
    if len(road_indices) > 0:
        large_monthly.loc[road_indices[0], "구분"] = "최초"
        if len(road_indices) > 1:
            large_monthly.loc[road_indices[1:], "구분"] = "지속"

# 메타 데이터(서비스처리센터, 행정동) 병합
def get_mode(x):
    return x.mode()[0] if not x.mode().empty else ""
road_meta = df_target.groupby("road_part").agg({
    "서비스처리센터": get_mode,
    "행정동": get_mode
}).reset_index()

final_result = pd.merge(large_monthly, road_meta, on="road_part", how="left")
final_result


,년도월,road_part,count,구분,서비스처리센터,행정동
0,2025-06,계룡로 520 (탄방동 둔산자이아이파크),90,최초,H071,탄방동
1,2025-07,계룡로 520 (탄방동 둔산자이아이파크),831,지속,H071,탄방동
2,2025-08,계룡로 520 (탄방동 둔산자이아이파크),700,지속,H071,탄방동
3,2025-09,계룡로 520 (탄방동 둔산자이아이파크),195,지속,H071,탄방동
4,2025-10,계룡로 520 (탄방동 둔산자이아이파크),48,지속,H071,탄방동
...,...,...,...,...,...,...
224,2025-08,탄방로 132 (용문동 둔산더샵엘리프),19,지속,H073,용문동
225,2025-11,탄방로 132 (용문동 둔산더샵엘리프),15,지속,H073,용문동
226,2024-01,태평동 334-15,42,최초,H073,태평동
227,2024-07,홍도동 65-2,46,최초,H072,홍도동


In [15]:
import os
# exports 폴더 생성
os.makedirs("exports", exist_ok=True)

# 신규아파트_오더 파일 출력
final_result.to_excel("exports/신규아파트_오더.xlsx", index=False)
print("[Success] exports/신규아파트_오더.xlsx 저장 완료")

# 1. 24~26년 신규 아파트 오더(15건 이상만) 행정동 월별 피벗 생성
# final_result 자체가 개별 단지별 15건 이상인 신규 오더만 있는 데이터프레임입니다.
# 따라서 df_final_matched를 거치는 대신 final_result를 직접 피벗하여 15건 미만 소량 오더를 자연스럽게 0으로 제거합니다.
pivot_report = final_result.pivot_table(
    index=["서비스처리센터", "행정동"],
    columns="년도월",
    values="count",
    aggfunc="sum",
    fill_value=0
)

# 2. 전체 데이터에서 서비스처리센터와 행정동의 유효 조합 추출하여 모든 행정동 리인덱스
df_all = df.copy()
df_all["서비스처리센터"] = df_all["서비스처리센터"].fillna("").astype(str).str.strip()
df_all["행정동"] = df_all["행정동"].fillna("").astype(str).str.strip()
df_all["서비스처리센터"] = df_all["서비스처리센터"].replace({"H051": "H073"})

df_all = df_all[
    (df_all["서비스처리센터"] != "") &
    (df_all["행정동"] != "") &
    (df_all["서비스처리센터"] != "nan") &
    (df_all["행정동"] != "nan")
]

all_indices = df_all.groupby(["서비스처리센터", "행정동"]).size().index

# 모든 행정동이 출력될 수 있도록 reindex 수행 (오더가 전혀 없는 곳은 0으로 채워짐)
pivot_report = pivot_report.reindex(all_indices, fill_value=0)

# 24~26년 전체 연월 목록으로 reindex (연월 열 정렬)
all_months = sorted(df_target["년도월"].dropna().unique())
pivot_report = pivot_report.reindex(columns=all_months, fill_value=0)

# 3. 최초 입주가 발생한 년도월/행정동 셀만 노란색 배경 칠하기 스타일러 정의
initial_apts = final_result[final_result["구분"] == "최초"]

def highlight_initial_apts(data):
    style_df = pd.DataFrame('', index=data.index, columns=data.columns)
    for idx, row in initial_apts.iterrows():
        center = row["서비스처리센터"]
        dong = row["행정동"]
        ym = row["년도월"]
        
        key = (center, dong)
        if key in data.index and ym in data.columns:
            style_df.loc[key, ym] = 'background-color: #FFFF99; font-weight: bold;'
    return style_df

# 스타일 적용 및 엑셀 저장 (Jinja2 부재 시 일반 엑셀 저장으로 우회 처리)
# try:
#     styled_pivot = pivot_report.style.apply(highlight_initial_apts, axis=None)
#     styled_pivot.to_excel("신규아파트_피벗리포트.xlsx")
#     print("[Success] exports/신규아파트_피벗리포트.xlsx (색상 하이라이트 적용 버전) 저장 완료")
# except ImportError:
#     pivot_report.to_excel("신규아파트_피벗리포트.xlsx")
#     print("[Success] exports/신규아파트_피벗리포트.xlsx (Jinja2 미설치로 일반 버전) 저장 완료")

# 피벗 리포트 출력
pivot_report


[Success] exports/신규아파트_오더.xlsx 저장 완료


년도월          2024-01  2024-02  2024-03  2024-04  2024-05  2024-06  2024-07  \
서비스처리센터 행정동                                                                  
H061    둔산동        0        0        0        0        0        0        0   
        월평동        0        0        0        0        0        0        0   
H071    가정동        0        0        0        0        0        0        0   
        갈마동        0        0        0        0        0        0        0   
        도룡동        0        0        0        0        0        0        0   
...              ...      ...      ...      ...      ...      ...      ...   
H075    중촌동        0        0        0        0        0        0        0   
        지족동        0        0        0        0        0        0        0   
        추목동        0        0        0        0        0        0        0   
        하기동        0        0        0        0        0        0        0   
        학하동        0        0        0        0        0        0        0   

년도월          2024-08  2024-09  2024-10  2024-11  2024-12  2025-01  2025-02  \
서비스처리센터 행정동                                                                  
H061    둔산동        0        0        0        0        0        0        0   
        월평동        0        0        0        0        0        0        0   
H071    가정동        0        0        0        0        0        0        0   
        갈마동        0        0        0        0        0        0        0   
        도룡동        0        0        0        0        0        0        0   
...              ...      ...      ...      ...      ...      ...      ...   
H075    중촌동        0        0        0        0        0        0        0   
        지족동        0        0        0        0        0        0        0   
        추목동        0        0        0        0        0        0        0   
        하기동        0        0        0        0        0        0        0   
        학하동        0        0        0        0        0        0        0   

년도월          2025-03  2025-04  2025-05  2025-06  2025-07  2025-08  2025-09  \
서비스처리센터 행정동                                                                  
H061    둔산동        0        0        0        0        0        0        0   
        월평동        0        0        0        0        0        0        0   
H071    가정동        0        0        0        0        0        0        0   
        갈마동        0        0        0        0        0        0        0   
        도룡동        0        0        0        0        0        0        0   
...              ...      ...      ...      ...      ...      ...      ...   
H075    중촌동        0        0        0        0        0        0        0   
        지족동        0        0        0        0        0        0        0   
        추목동        0        0        0        0        0        0        0   
        하기동        0        0        0        0        0        0        0   
        학하동        0        0        0        0        0        0        0   

년도월          2025-10  2025-11  2025-12  2026-01  2026-02  2026-03  2026-04  
서비스처리센터 행정동                                                                 
H061    둔산동        0        0        0        0        0        0        0  
        월평동        0        0        0        0        0        0        0  
H071    가정동        0        0        0        0        0        0        0  
        갈마동        0        0        0        0        0        0        0  
        도룡동        0        0        0        0        0        0        0  
...              ...      ...      ...      ...      ...      ...      ...  
H075    중촌동        0        0        0        0        0        0        0  
        지족동        0        0        0        0        0        0        0  
        추목동        0        0        0        0        0        0        0  
        하기동        0        0        0        0        0        0        0  
        학하동        0        0  

In [16]:
check_keyword = "위드힐"
# 1. 메모리에 구축된 2023년 Baseline 집합(addresses_2023)에서 검색
matched_in_2023 = [addr for addr in addresses_2023 if check_keyword in str(addr)]
print(f"🔎 === '{check_keyword}' 2023년 이력(Baseline) 검증 결과 ===")
if matched_in_2023:
    print(f"🚨 [이력 있음 ➡️ 신규 제외] 2023년 기존 이력에 존재하여 피벗 집계에서 제외(0건 처리)되었습니다.")
    print(f"   -> 2023년 Baseline 등록명: {matched_in_2023}")
else:
    print(f"🌟 [이력 없음 ➡️ 신규 통과] 2023년 이력이 전혀 없었던 순수한 신축 단지 주소입니다!")
# 2. 실제 2023년 원본 데이터에서 몇 건의 오더가 있었는지 날것(Raw)으로 교차 검증
raw_2023_count = df_matched[
    df_matched["년도월"].str.startswith("2023-", na=False) &
    df_matched["주소"].str.contains(check_keyword, na=False)
]
print(f"📊 2023년 실제 오더 접수 건수: {len(raw_2023_count)}건")

🔎 === '위드힐' 2023년 이력(Baseline) 검증 결과 ===
🚨 [이력 있음 ➡️ 신규 제외] 2023년 기존 이력에 존재하여 피벗 집계에서 제외(0건 처리)되었습니다.
   -> 2023년 Baseline 등록명: ['대전로542번길 121 (천동 540 위드힐)']
📊 2023년 실제 오더 접수 건수: 70건


### 8단계 : 2023년 신규 아파트 데이터 취합

In [17]:
# ──────────────────────────────────────────────────────────────────────────────
# [신규] 2023년 신규 아파트 오더 탐지 (표제부 사용승인일 기반)
# 사전 조건: CELL 21이 실행되어 df_matched, road_to_master 변수가 생성되어 있어야 함
# ──────────────────────────────────────────────────────────────────────────────

# ── STEP 1. 표제부 5개 파일 로드 ─────────────────────────────────────────────
표제부_경로 = {
    "대덕구": r"\\DocuONE\MyDrive\개인함\23_26년_서비스오더\표제부\대덕구_표제부.xlsx",
    "동구":   r"\\DocuONE\MyDrive\개인함\23_26년_서비스오더\표제부\동구_표제부.xlsx",
    "서구":   r"\\DocuONE\MyDrive\개인함\23_26년_서비스오더\표제부\서구_표제부.xlsx",
    "유성구": r"\\DocuONE\MyDrive\개인함\23_26년_서비스오더\표제부\유성구_표제부.xlsx",
    "중구":   r"\\DocuONE\MyDrive\개인함\23_26년_서비스오더\표제부\중구_표제부.xlsx",
}

In [18]:
def extract_road_key_from_full(full_addr):
    """
    "대전광역시 서구 계룡로 520 (탄방동)" → "계룡로 520"
    도로명대지위치 전체주소에서 시/구 제거 후 도로명+번호만 반환
    """
    if pd.isna(full_addr):
        return ""
    s = str(full_addr).split("(")[0].strip()
    parts = s.split()
    for i, p in enumerate(parts):
        if re.search(r"(로|길|가|대로)\b", p) or re.search(r"(로|길|가)\d+", p):
            return " ".join(parts[i:]).strip()
    return " ".join(parts[-2:]).strip() if len(parts) >= 2 else s

In [19]:
pyo_dfs = []
for gu_name, path in 표제부_경로.items():
    try:
        _df = pd.read_excel(path)
        _df["구"] = gu_name
        pyo_dfs.append(_df)
        print(f"  [{gu_name}] {len(_df):,}행 로드 완료")
    except Exception as e:
        print(f"  [경고] {gu_name} 로드 실패: {e}")

df_pyo_all = pd.concat(pyo_dfs, ignore_index=True)
df_pyo_all["사용승인일"] = df_pyo_all["사용승인일"].astype(str).str.strip()
df_pyo_all["road_key"] = df_pyo_all["도로명대지위치"].apply(extract_road_key_from_full)

  [대덕구] 22,185행 로드 완료
  [동구] 28,931행 로드 완료
  [서구] 29,055행 로드 완료
  [유성구] 25,526행 로드 완료
  [중구] 36,285행 로드 완료


In [20]:
# 사용승인일 날짜 정제 함수 (앞 4자리 연도만 추출)
def clean_date_to_year(val):
    if pd.isna(val):
        return ""
    val_str = str(val).strip()
    if val_str.endswith(".0"):
        val_str = val_str[:-2]
    val_str = re.sub(r"\D", "", val_str) # 숫자만 남기기
    if len(val_str) >= 4:
        return val_str[:4] # 앞 4자리(연도) 추출
    return ""

df_pyo_all["사용승인년도"] = df_pyo_all["사용승인일"].apply(clean_date_to_year)

# 유효한 행만 (road_key 있고 사용승인년도 4자리 숫자)
df_pyo_valid = df_pyo_all[
    (df_pyo_all["road_key"] != "") &
    (df_pyo_all["사용승인년도"].str.len() == 4)
].copy()

# road_key 기준 최신 사용승인년도 (마지막 동 기준)
road_approval_map = (
    df_pyo_valid.groupby("road_key")["사용승인년도"]
    .max()
    .to_dict()
)
print(f"\n표제부 road_key → 사용승인년도 매핑: {len(road_approval_map):,}개")



표제부 road_key → 사용승인일 매핑: 73,517개


In [21]:
# -- STEP 2. road_to_master의 각 대표(canonical) 주소를 표제부와 연결 --------------
master_to_approval = {}
for master_full_addr in set(road_to_master.values()):
    road_key = master_full_addr.split('(')[0].strip()
    if road_key in road_approval_map:
        master_to_approval[master_full_addr] = road_approval_map[road_key]
print(f"마스터 단지 중 표제부 매칭 성공: {len(master_to_approval):,}개")

# -- STEP 3. 2023년 데이터 추출 + 연도월 컬럼명 자동 감지 ---------------
print("df_matched 컬럼:", list(df_matched.columns))
ym_col = None
for _cand in ["연도월", "년도월"]:
    if _cand in df_matched.columns:
        ym_col = _cand
        break
if ym_col is None:
    raise KeyError(f"연도월/년도월 컬럼 없음. 실제 컬럼: {list(df_matched.columns)}")
print(f"사용할 연도월 컬럼명: '{ym_col}'")

# df_filtered에서 2023년 데이터 추출
df_2023_all = df_filtered[df_filtered[ym_col].str.startswith("2023-")].copy()

# -- STEP 4. [개선] 연도월 x 주소_정제 기준 15건 이상 필터 ---------------------
df_2023_all["addr_count"] = df_2023_all.groupby([ym_col, "주소_정제"])["주소_정제"].transform("size")
df_2023_large = df_2023_all[df_2023_all["addr_count"] >= 15].copy()

# -- STEP 5. confirmed_apartments 리스트 매칭 ------------------------------------
df_2023_large["road_part"] = df_2023_large["주소_키워드"].map(road_to_master)
df_2023_matched = df_2023_large[df_2023_large["road_part"].notna()].copy()

# -- STEP 6. 표제부 사용승인일 매칭 및 2022~2023 필터 -------------------------
df_2023_matched["사용승인일"] = df_2023_matched["road_part"].map(master_to_approval)
new_mask = (
    df_2023_matched["사용승인일"].notna() &
    df_2023_matched["사용승인일"].isin(["2022", "2023"])
)
df_2023_new_details = df_2023_matched[new_mask].copy()

# 월별/단지별 최종 집계
df_2023_new = (
    df_2023_new_details.groupby([ym_col, "road_part"])
    .size()
    .reset_index(name="count")
)
print(f"신규 아파트 필터 결과: {len(df_2023_new)}건 / 고유 단지: {df_2023_new['road_part'].nunique()}개")

# -- STEP 7. 최초/지속 라벨링 ------------------------------------------
df_2023_new = df_2023_new.sort_values(by=["road_part", ym_col])
df_2023_new["구분"] = ""
for road in df_2023_new["road_part"].unique():
    idx = df_2023_new[df_2023_new["road_part"] == road].index
    if len(idx) > 0:
        df_2023_new.loc[idx[0], "구분"] = "최초"
        if len(idx) > 1:
            df_2023_new.loc[idx[1:], "구분"] = "지속"

# -- STEP 8. 처리센터/행정동 메타 병합 ----------------------------------
def _get_mode(x):
    return x.mode()[0] if not x.mode().empty else ""

road_meta_2023 = df_2023_new_details.groupby("road_part").agg(
    서비스처리센터=("서비스처리센터", _get_mode),
    행정동=("행정동", _get_mode),
).reset_index()

df_2023_new = pd.merge(df_2023_new, road_meta_2023, on="road_part", how="left")

# -- STEP 9. 원본 데이터 저장 (안전하게 최종 접미사 붙여서 저장) ---------------
df_2023_new["사용승인일_포맷"] = df_2023_new["road_part"].map(master_to_approval)
out_cols = [ym_col, "road_part", "사용승인일_포맷", "count", "구분", "서비스처리센터", "행정동"]
try:
    df_2023_new[out_cols].to_excel("exports/2023년_신규아파트오더_최종.xlsx", index=False)
    print("\n[저장 완료] exports/2023년_신규아파트오더_최종.xlsx")
except PermissionError:
    df_2023_new[out_cols].to_excel("exports/2023년_신규아파트오더_최종_대체.xlsx", index=False)
    print("\n[경고: 권한오류] 대안 경로로 저장 완료: exports/2023년_신규아파트오더_최종_대체.xlsx")

df_2023_new


마스터 단지 중 표제부 매칭 성공: 391개
df_matched 컬럼: ['서비스처리센터', '고객방문일', '고객방문시간', '우선순위', '오더번호', '상태', '대분류', '중분류', '소분류', '내역', '세대', '고객서비스처리센터', '고객번호', '계약계정번호', '정보', '정보(BP)', '최초공급일', '주소', '건물동', '호수', '전입/전출일', '오더생성일', '오더생성시간', '오더생성자', '부서확인일', '부서확인시간', '부서확인자', '기사배정일', '기사배정시간', '배정된기사', '마감일', '마감시간', '마감자', '변경일', '변경시간', '변경자', '수수료금액', '영수증번호', '행정동', '시트명', 'Unnamed: 10', '세대.1', '설치유형', '설치유형 Text', '년도월', '주소_정제', '주소_키워드', 'road_part']
사용할 연도월 컬럼명: '년도월'
2023년 전입신청/변경 전체 건수: 40,927건
15건 이상 조합: 557개 / 고유 단지: 188개
  - 매칭 성공: 307개
  - 미매칭:    250개
신규 아파트 필터 결과: 64건 / 고유 단지: 30개

[저장 완료] exports/2023년_신규아파트오더.xlsx


,년도월,road_part,사용승인일_포맷,count,구분,서비스처리센터,행정동
0,2023-09,계룡로921번길 43 (대사동),2023-09-14,21,최초,H074,대사동
1,2023-09,계족로409번길 16 (홍도동),2023-10-18,22,최초,H072,홍도동
2,2023-01,계족로517번길 25 (용전동),2022-12-14,24,최초,H072,용전동
3,2023-01,과학성장로 33 (둔곡동),2022-11-07,213,최초,H072,둔곡동
4,2023-02,과학성장로 33 (둔곡동),2022-11-07,56,지속,H072,둔곡동
5,2023-03,과학성장로 33 (둔곡동),2022-11-07,54,지속,H072,둔곡동
6,2023-04,과학성장로 33 (둔곡동),2022-11-07,24,지속,H072,둔곡동
7,2023-01,과학성장로 77 (둔곡동 430 서한이다음1단지),2022-11-04,208,최초,H072,둔곡동
8,2023-02,과학성장로 77 (둔곡동 430 서한이다음1단지),2022-11-04,98,지속,H072,둔곡동
9,2023-03,과학성장로 77 (둔곡동 430 서한이다음1단지),2022-11-04,66,지속,H072,둔곡동


### 8-1단계: 2023년 신규 아파트 월별 피벗 테이블 생성

* df_2023_new 를 서비스처리센터 x 행정동 행, 월별 열로 피벗
* count = 신규 단지에서 발생한 전입 건수 (15건 이상 단지만 반영)

In [22]:
# -- 8-1단계: 피벗 테이블 생성 ------------------------------------------
# 사전 조건: CELL 32 실행 완료 (df_2023_new, ym_col 변수 필요)

# 2023년 전체 12개월 고정
ALL_2023_MONTHS = [f"2023-{m:02d}" for m in range(1, 13)]

# 서비스처리센터 x 행정동 x 연도월 피벗 (count 합산)
pivot_2023 = df_2023_new.pivot_table(
    index=["서비스처리센터", "행정동"],
    columns=ym_col,
    values="count",
    aggfunc="sum",
    fill_value=0
)

# 2023년 전체 월로 컬럼 정렬 (없는 달도 0으로 채움)
pivot_2023 = pivot_2023.reindex(columns=ALL_2023_MONTHS, fill_value=0)

print(f"피벗 생성 완료: {pivot_2023.shape[0]}행 x {pivot_2023.shape[1]}열")
pivot_2023


피벗 생성 완료: 21행 x 12열


년도월           2023-01  2023-02  2023-03  2023-04  2023-05  2023-06  2023-07  \
서비스처리센터 행정동                                                                   
H072    둔곡동       468      187      147       24       21        0        0   
        신탄진동       24        0        0        0        0        0        0   
        용산동         0        0        0      840      479      312       67   
        용전동        24        0        0        0        0        0        0   
        중리동        22        0        0        0        0        0        0   
        홍도동       114      149      101       26       16        0        0   
H073    유천동         0        0        0        0        0        0        0   
H074    가양동       104       45       27       18        0        0        0   
        대사동         0        0        0        0        0        0        0   
        대흥동         0        0        0        0        0        0        0   
        목동         16       21        0        0        0        0        0   
        문창동         0        0        0        0        0        0        0   
        문화동         0       20        0        0        0        0        0   
        신흥동        19       21        0        0        0        0        0   
        판암동        29        0        0        0        0        0        0   
H075    궁동         17       43        0        0        0        0        0   
        덕명동        26        0        0        0        0        0        0   
        복용동         0       29        0        0        0        0        0   
        봉명동         0       59        0        0        0        0        0   
        원신흥동        0        0        0        0        0        0        0   
        장대동        18       20        0       20        0        0        0   

년도월           2023-08  2023-09  2023-10  2023-11  2023-12  
서비스처리센터 행정동                                                
H072    둔곡동         0       45        0       48        0  
        신탄진동        0        0        0        0        0  
        용산동        37        0        0        0        0  
        용전동         0        0       20        0        0  
        중리동         0        0        0        0        0  
        홍도동         0       22        0        0        0  
H073    유천동         0        0        0        0       61  
H074    가양동         0        0        0        0        0  
        대사동         0       21        0        0        0  
        대흥동         0        0        0       22        0  
        목동          0        0        0       27       15  
        문창동         0        0        0       27        0  
        문화동         0        0        0        0        0  
        신흥동         0        0        0        0        0  
        판암동         0        0        0        0        0  
H075    궁동          0        0        0        0        0  
        덕명동         0        0        0        0        0  
        복용동         0        0       18        0       16  
        봉명동         0        0        0       42        0  
        원신흥동        0        0      209      230      111  
        장대동         0        0        0        0        0

### 8-2단계: 전체 행정동 reindex (0건 행정동도 표시)

* 전체 df 기준 유효한 (서비스처리센터, 행정동) 조합으로 reindex
* 신규 오더가 없는 행정동도 0으로 채워 빠짐없이 표시

In [23]:
# -- 8-2단계: 전체 행정동 reindex (0건 행정동도 포함) --------------------

# 전체 df에서 유효한 (서비스처리센터, 행정동) 쌍 추출
df_all_clean = df.copy()
df_all_clean["서비스처리센터"] = (
    df_all_clean["서비스처리센터"]
    .fillna("").astype(str).str.strip()
    .replace({"H051": "H073"})
)
df_all_clean["행정동"] = df_all_clean["행정동"].fillna("").astype(str).str.strip()

# 빈값 및 nan 제거
df_all_clean = df_all_clean[
    (df_all_clean["서비스처리센터"] != "") &
    (df_all_clean["서비스처리센터"] != "nan") &
    (df_all_clean["행정동"] != "") &
    (df_all_clean["행정동"] != "nan")
]

# 전체 (서비스처리센터, 행정동) MultiIndex
all_valid_index = df_all_clean.groupby(["서비스처리센터", "행정동"]).size().index

# 피벗을 전체 행정동으로 확장 (신규 오더 없는 곳 = 0)
pivot_2023 = pivot_2023.reindex(all_valid_index, fill_value=0)

print(f"reindex 후: {pivot_2023.shape[0]}행 (전체 행정동 포함)")
print(f"신규 오더 발생 행 수: {(pivot_2023.sum(axis=1) > 0).sum()}")


reindex 후: 141행 (전체 행정동 포함)
신규 오더 발생 행 수: 21


### 8-3단계: 최초 발생 셀 하이라이트 및 엑셀 저장

* 최초(첫 번째 월) 신규 오더 셀 -> #FFFF99 배경 + 볼드
* exports/2023년_신규아파트_피벗.xlsx 저장

In [24]:
# -- 8-3단계: 아파트오더가 있는 행만 필터링 및 최초 발생 셀 하이라이트(#FFFF99+볼드) 및 엑셀 저장 --------
import openpyxl
from openpyxl.styles import PatternFill, Font

# 구분 == 최초 인 행들 (신규 단지의 첫 번째 전입 월)
initial_apts_2023 = df_2023_new[df_2023_new["구분"] == "최초"].copy()

# 아파트오더가 있는 행만 필터링 (행의 합계가 0보다 큰 행)
pivot_2023_filtered = pivot_2023[pivot_2023.sum(axis=1) > 0].copy()

output_path = "exports/2023년_신규아파트_피벗_최종.xlsx"

try:
    pivot_2023_filtered.to_excel(output_path)
    print(f"[저장 완료] {output_path}")
except PermissionError:
    output_path = "exports/2023년_신규아파트_피벗_최종_대체.xlsx"
    pivot_2023_filtered.to_excel(output_path)
    print(f"[경고: 권한오류] 대안 경로로 저장 완료: {output_path}")

try:
    wb = openpyxl.load_workbook(output_path)
    ws = wb.active
    
    yellow_fill = PatternFill(start_color="FFFF99", end_color="FFFF99", fill_type="solid")
    bold_font = Font(bold=True)
    
    col_mapping = {}
    for col_idx in range(3, ws.max_column + 1):
        month_val = ws.cell(row=1, column=col_idx).value
        if month_val:
            col_mapping[str(month_val).strip()] = col_idx
            
    for _, row in initial_apts_2023.iterrows():
        center = str(row["서비스처리센터"]).strip()
        dong   = str(row["행정동"]).strip()
        ym     = str(row[ym_col]).strip()
        
        target_row = None
        for r in range(2, ws.max_row + 1):
            cell_center = str(ws.cell(row=r, column=1).value).strip()
            cell_dong   = str(ws.cell(row=r, column=2).value).strip()
            if cell_center == center and cell_dong == dong:
                target_row = r
                break
                
        if target_row and ym in col_mapping:
            target_col = col_mapping[ym]
            ws.cell(row=target_row, column=target_col).fill = yellow_fill
            ws.cell(row=target_row, column=target_col).font = bold_font
            
    wb.save(output_path)
    print(f"[스타일 적용 완료] openpyxl 기반 스타일링이 성공적으로 입혀졌습니다.")
except Exception as e:
    print(f"[스타일 경고] openpyxl 스타일 입히기 실패: {e}")

print(f"  - 전체 행 수 (필터 후): {pivot_2023_filtered.shape[0]}")
print(f"  - 컬럼(월): {list(pivot_2023_filtered.columns)}")
pivot_2023_filtered


[저장 완료] exports/2023년_신규아파트_피벗.xlsx
  - 최초 하이라이트 셀 수: 30개
  - 전체 행 수: 141
  - 컬럼(월): ['2023-01', '2023-02', '2023-03', '2023-04', '2023-05', '2023-06', '2023-07', '2023-08', '2023-09', '2023-10', '2023-11', '2023-12']
